# Exploring manually labelled dataset

In [ ]:
# Imports
from typing import Final
from pathlib import Path
from os import getenv
from dotenv import find_dotenv, load_dotenv
import re
import numpy as np
import shapely as sh
import rasterio
import pandas as pd
import geopandas as gp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import pairwise_distances
from outputs import convert_ToponymExtractor_outputs_to_gdf
from edina import get_transformer_from_geodataframe

PROJECT_DIR: Final[Path] = Path(find_dotenv(".env", 1, 1)).absolute().parent
load_dotenv(PROJECT_DIR.joinpath(".env"))
LOCAL_DIR: Final[Path] = Path(getenv("LOCAL_DIR"))

In [ ]:
# load dataset from which manual labelling was derived
source = gp.read_file(LOCAL_DIR.joinpath(
    "outputs/toponym-extractor/revised/glam-st17ne-2.gpkg"
))
# load manually labelled dataset
gdf = gp.read_file(LOCAL_DIR.joinpath(
    "outputs/manual-labelling/raw/glamst17ne2-manually-labelled.gpkg"
))
# Make sure all status characters are uppercase
gdf["status"] = gdf.status.str.upper()
gdf.head()

In [ ]:
# load refined manually labelled dataset
refined = gp.read_file(LOCAL_DIR.joinpath(
    "outputs/manual-labelling/processed/glamst17ne2-manually-labelled.gpkg"
))

## Top level summary

In [ ]:
n_preds = len(source)
n_pred_groups = len(source[["png_filename", "groupid"]].drop_duplicates())

n_preds_verified = (
    ((~gdf.status.str.contains("N")) | (gdf.status == "DNE"))
    & (gdf.verified)
)
n_preds_verified = sum(
    1
    for multi in gdf.loc[n_preds_verified, "geometry"].array
    for _ in multi.geoms
)

n_groups_verified = ((~gdf.status.str.contains("N")) & (gdf.verified))
n_groups_verified = len(
    gdf.loc[n_groups_verified, ["png_filename", "groupid"]].drop_duplicates()
)

selection = (
    gdf.status.str.contains("N")
    & (gdf.status != "DNE")
    & (~gdf.word.str.contains("$", regex = False))
    & (gdf.verified)
)
n_new_labels = selection.sum()

n_new_groups = len(
    gdf.loc[selection, ["png_filename", "groupid"]].drop_duplicates()
)

selection = (
    (~gdf.status.str.contains("N", regex = False))
    & (gdf.word.str.contains("$", regex = False))
    & (gdf.verified)
)
n_non_words_detected =\
    (((gdf.status == "DNE") & (gdf.verified)) | selection).sum()

selection = (gdf.geometry.buffer(0).geom_type == "MultiPolygon")
n_words_merged = sum(
    1
    for multi in gdf.loc[selection, "geometry"].array
    for _ in multi.geoms
)

selection = (
    (gdf.status.str.contains("O", regex = False))
    &(~gdf.status.str.contains("N", regex = False))
    &(~gdf.status.str.contains("U", regex = False))
    & gdf.verified
)
n_words_overlapped = selection.sum()

selection = (
    (gdf.status.str.contains("U", regex = False))
    & (~(gdf.geometry.buffer(0).geom_type == "MultiPolygon"))
    &(~gdf.status.str.contains("N", regex = False))
    & gdf.verified
)
n_preds_unidentifiable = selection.sum()

selection = ((gdf.status == "NU") & gdf.verified)
n_manually_unidentifiable = selection.sum()

selection = ((gdf.status == "C") & gdf.verified)
n_words_correct = selection.sum()

selection = (
    gdf.verified
    & (~(gdf.geometry.buffer(0).geom_type == "MultiPolygon"))
    & (
        (
            gdf.status.str.contains("E", regex = False) 
            & (~gdf.status.str.contains("O", regex = False))
            & (~gdf.status.str.contains("U", regex = False))
            & (~gdf.status.str.contains("N", regex = False))
        )
        | (gdf.status == "GWP")
    )
)
n_words_edited = selection.sum()

print(
    f"Number of word instances predicted: {n_preds:,}",
    f"Grouped into {n_pred_groups} toponyms",
    f"Number of word predictions verified: {n_preds_verified:,} ({100 * n_preds_verified / n_preds:.1f}% of detected word instances)",
    f"Consolidated into {n_groups_verified} toponyms",
    f"Number of new word instances manually discovered: {n_new_labels:,}",
    f"Grouped into {n_new_groups} new toponyms",
    f"Number of map features misinterpreted as words: {n_non_words_detected:,}",
    f"Number of word instances combined into a single instance: {n_words_merged:,}",
    f"Number of word instances suppressed due to overlapping another instance: {n_words_overlapped}",
    f"Number of word predictions unidentifiable: {n_preds_unidentifiable}",
    f"Number of unidentifiable words manually discovered: {n_manually_unidentifiable}",
    f"Number of word instances predicted correctly: {n_words_correct}",
    f"Number of predicted words requiring edits: {n_words_edited}",
    sep = "\n"
)

Visuals:

Chart 1:

Bar chart:

- bar 1: (greyed out) Number of word instances predicted
- bar 2: Number of manual labels (preds verified + manually disc.)
- bar 3: Number of predicted word instances verified
- bar 4: (offset by number of word instances verified) Number of manual labels

Chart 2:

Horizontal bar chart:

- bar 1: (greyed out) Number of predicted word instances verified
- bar 2: Number of correct predictions
- bar 3: (offset stacked from bar 2) Number of predictions requiring edits
- bar 4: (offset stacked from bar 2 + bar 3) Number of predictions requiring edits
- bar 5: (offset stacked from bar 2 + bar 3 + bar 4) Combined into a single instance
- bar 6: (offset stacked from bar 2 + ... + bar 5) Unidentifiable predicted words
- bar 7: (offset stacked from bar 2 + ... + bar 6) Features misinterpreted as words
- bar 8: (offset stacked from bar 2 + ... + bar 7) Suppressed do to overlap

Chart 3:

Cardiff map with bounding boxes showing which area is reviewed and which isn't.


## Calculate timescales

Summing up amount of time each day spent manually labelling

In [ ]:
times = [
    pd.Timedelta("7hr28min"),
    pd.Timedelta("9hr35min"),
    pd.Timedelta("5hr17min"),
    pd.Timedelta("5hr54min"),
    pd.Timedelta("10hr32min"),
    pd.Timedelta("8hr6min"),
    pd.Timedelta("3hr29min"),
]
total_time = sum(times, start = pd.Timedelta(0))
total_time

In [ ]:
total_time / (n_preds_verified + n_new_labels)

~70 seconds per word label.

## Check group distances
Using refined labels

In [ ]:
# Get toponyms with more than one word instance
multi_word_topo = refined\
    .groupby(["png_filename", "groupid"], as_index = False)\
    .agg(word_count = ("wordid", "count"))
multi_word_topo = multi_word_topo[multi_word_topo.word_count > 1]

# Generate max distances by cycling through each multi-word toponym record
max_dist = pd.Series([])
for topo in multi_word_topo.itertuples(index = True):
    # get distance matrix for word masks belonging to toponym
    selection = (
        (refined.png_filename == topo.png_filename)
        & (refined.groupid == topo.groupid)
    )
    selection = pairwise_distances(
        X = refined.loc[selection, "geometry"], metric = sh.distance
    )
    # pass max distance to series
    max_dist[topo.Index] = selection.max(axis = None)

multi_word_topo["max_dist"] = max_dist

In [ ]:
# temp - mark to double check
multi_word_topo["check"] = False
multi_word_topo.loc[multi_word_topo.max_dist > 50, "check"] = True

refined_check = pd.merge(
    left = refined,
    right = multi_word_topo[["png_filename", "groupid", "check"]],
    how = "left",
    on = ["png_filename", "groupid"]
)
refined_check["check"] = refined_check.check.fillna(False)
refined_check.head()

In [ ]:
refined_check.to_file(LOCAL_DIR.joinpath(
    "outputs/manual-labelling/glamst17ne2-manually-double-check.gpkg"
))

## Check overlap IoU, A in B, B in A

In [ ]:
overlap = gdf[gdf.status.str.contains("O")]
gdf_overlap = gp.sjoin(gdf, overlap)
gdf_overlap = gdf_overlap[gdf_overlap.index != gdf_overlap.index_right]
gdf_overlap["intersect_area"] = gdf_overlap\
    .geometry\
    .intersection(overlap.loc[gdf_overlap.index_right, "geometry"], align = 0)\
    .area

gdf_overlap["l_in_r_area_pc"] =\
    gdf_overlap.intersect_area / gdf_overlap.geometry.area

gdf_overlap["r_in_l_area_pc"] = (
    gdf_overlap.intersect_area
    / overlap.loc[gdf_overlap.index_right, "geometry"].area.array
)

gdf_overlap["iou"] = gdf_overlap\
    .geometry\
    .union(overlap.loc[gdf_overlap.index_right, "geometry"], align = 0)\
    .area
gdf_overlap["iou"] = gdf_overlap.intersect_area / gdf_overlap.iou

overlap_cols = ["l_in_r_area_pc", "r_in_l_area_pc", "iou"]
overlap_summary = gdf_overlap[["index_right", *overlap_cols]]\
    .groupby(by = "index_right", as_index = False)\
    .max()

In [ ]:
overlap_summary

In [ ]:
overlap_summary[["l_in_r_area_pc", "r_in_l_area_pc", "iou"]].plot.box()

In [ ]:
overlap_summary[overlap_cols].plot.hist(bins = 100)

In [ ]:
overlap_summary\
    .loc[(overlap_summary.r_in_l_area_pc < .8), overlap_cols]\
    .plot\
    .hist(bins = 100)

In [ ]:
selection = (
    (overlap_summary.r_in_l_area_pc < .6)
    & (overlap_summary.l_in_r_area_pc < .8)
)
overlap_summary\
    .loc[selection, overlap_cols]\
    .plot\
    .hist(bins = 80)

## Visualize Edits

In [ ]:
from matplotlib import colors
from numpy.typing import ArrayLike

def desaturate_colour(rgb: ArrayLike, satfrac: float = .5) -> np.ndarray:
    """
    Desaturates colour by fraction given.

    Parameters
    ----------
    rgb: ArrayLike of size 3.
        Required. ArrayLike of red, green, and blue values. Each value
        must be in range [0., 1.].

    satfrac: float. Default: 0.5.
        Optional. proportion to reduce saturation by. If no argument is
        given, then default is 0.5

    Returns
    -------
    Numpy array: Array of rgb values.
    """
    # check satfrac is in range [0., 1.]
    if (satfrac > 1.) or (satfrac < 0.):
        raise ValueError(
            f"satfrac argument outside of accepted range; value must be "\
            f"between 0. and 1. (inclusive). Argument passed: {satfrac}"
        )
    hsv = colors.rgb_to_hsv(rgb)
    hsv[1] *= satfrac
    return colors.hsv_to_rgb(hsv)

In [ ]:
from collections import OrderedDict
from matplotlib import colors
from matplotlib.patches import Rectangle

# Construct bar plot
selection = (
    ~refined.status.str.contains("[NU]", regex = True)
    & ~refined.word.str.contains("$", regex = False)
)
edited_group = selection & refined.status.str.contains("G", regex = False)
edited_word = selection & refined.status.str.contains("W", regex = False)
edited_poly = selection & refined.status.str.contains("P", regex = False)

edited_counts = OrderedDict((
    ("Retained Labels", selection.sum()),
    ("Edited Labels", (edited_group | edited_word | edited_poly).sum()),
    ("Edited Group", edited_group.sum()),
    ("Edited Word", edited_word.sum()),
    ("Edited Polygon", edited_poly.sum())
))
bar_labels = [f"{edited_counts["Retained Labels"]:,}", *(
    f"{v:,}\n({100 * v / edited_counts["Retained Labels"]:.1f}%*)"
    for k, v in edited_counts.items()
    if k != "Retained Labels"
)]
face_colours = {
    "Retained Labels":  desaturate_colour(colors.to_rgb("tab:blue"), .25),
    "Edited Labels": "tab:blue",
    "Edited Group": "cornflowerblue",
    "Edited Word": "cornflowerblue",
    "Edited Polygon": "cornflowerblue"
}
edge_colours = {
    "Retained Labels": desaturate_colour(colors.to_rgb("tab:blue"), .25),
    "Edited Labels": "gold",
    "Edited Group": "gold",
    "Edited Word": "gold",
    "Edited Polygon": "gold"
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (18, 8), layout = "constrained")
barch = ax1.bar(
    x = [*edited_counts.keys()],
    height = [*edited_counts.values()],
    fc = [face_colours[k] for k in edited_counts.keys()],
    ec = [edge_colours[k] for k in edited_counts.keys()],
    lw = 1
)
barlabs = ax1.bar_label(
    barch,  bar_labels, label_type = "center", fontsize = 12
)
_ = ax1.set_title("Edit counts in retained labels", fontsize = 14)
_ = ax1.grid(True, axis = "y"),ax1.set_axisbelow(True),ax1.set_ylabel("Count")

# Construct heatmap
heat_idx = pd.Index(["Group", "Polygon", "Word", "Group, polygon & word"])
group_col = pd.Series(
    data = [
        (edited_group & ~edited_word & ~edited_poly).sum(),
        (edited_group & ~edited_word & edited_poly).sum(),
        (edited_group & edited_word & ~edited_poly).sum(),
        np.nan
    ],
    index = heat_idx,
    name = "Group"
)
poly_col = pd.Series(
    data = [
        (edited_group & ~edited_word & edited_poly).sum(),
        (~edited_group & ~edited_word & edited_poly).sum(),
        (~edited_group & edited_word & edited_poly).sum(),
        np.nan
    ],
    index = heat_idx,
    name = "Polygon"
)
word_col = pd.Series(
    data = [
        (edited_group & edited_word & ~edited_poly).sum(),
        (~edited_group & edited_word & edited_poly).sum(),
        (~edited_group & edited_word & ~edited_poly).sum(),
        np.nan
    ],
    index = heat_idx,
    name = "Word"
)
gpw_col = pd.Series(
    data = [
        np.nan,
        np.nan,
        np.nan,
        (edited_group & edited_word & edited_poly).sum()
    ],
    index = heat_idx,
    name = "Group, polygon & word"
)
edits = pd.DataFrame([group_col, poly_col, word_col, gpw_col])

# Construct annotations
edits_anno = edits.astype("Int64").astype("str") + "\n"
edits_anno += (100 * edits / edited_counts["Retained Labels"])\
    .round(1).astype("str")
edits_anno += "%*"
edits_anno = edits_anno.fillna("")

# build visual
sns.heatmap(
    data = edits,
    mask = np.tri(len(edits), k = -1, dtype = "bool").T,
    square = True,
    vmax = edits.max(axis = None),
    vmin = 0,
    cmap = "Blues",
    cbar_kws = {"label": "Counts", "shrink": 1.},
    annot = edits_anno.to_numpy(),
    annot_kws = {"size": 12},
    fmt = "",
    ax = ax2,
    linewidths = .75
)
ax2.patch.set_facecolor('white')
ax2.patch.set_edgecolor('black')  # will be used for hatching
ax2.patch.set_hatch('/////')
ax2.add_artist(Rectangle((0., 0.), 1, 1, ec = "gold", lw = 3, fc = [0.] * 4))
ax2.add_artist(Rectangle((1., 1.), 1, 1, ec = "gold", lw = 3, fc = [0.] * 4))
ax2.add_artist(Rectangle((2., 2.), 1, 1, ec = "gold", lw = 3, fc = [0.] * 4))
ax2.collections[0].colorbar.outline.set_linewidth(1) # make outline visible
ax2.collections[0].colorbar.outline.set_edgecolor('black')
_ = ax2.set_xlabel("Feature edited"), ax2.set_ylabel("Feature edited")
_ = ax2.set_title(
    "Counts of intersecting edits\n(Gold boxed squares are counts of records "\
    "with only one feature edited)",
    fontsize = 14
)
plt.figtext(0.01, 0.01, '\n* Percent of retained labels', fontsize = 12)

Large number of group edits resulting partly from inconsistent group labelling when supressing overlapping word instances.

Extending retry predictions strategy for overlapping instances to groups containing words that have either an IoU or proportion of area intesecting at .5 or higher. No comparison against polygons not belonging to overlapping png regions was made here because false positives don't matter too much.

## Compare manual labels against Gb1900 gazetteer

In [ ]:
# read revised predictions
revised = gp.read_file(LOCAL_DIR.joinpath(
    "outputs/manual-labelling/revised/glamst17ne2-manually-double-checked.gpkg"
))
# Load GB1900 gazetteer
gb1900 = gp\
    .read_file(LOCAL_DIR.joinpath("outputs/pngs/text-locations.gpkg"))
gb1900 = gb1900[["pin_id", "tiff_filename", "final_text", "geometry"]]
gb1900 = gb1900.drop_duplicates()
gb1900 = gb1900[gb1900.tiff_filename == "glam-st17ne-2.tif"]
gb1900

In [ ]:
# Construct GeoDataFrame of toponyms
selection =\
    ((~revised.word.isna()) & (~revised.word.str.contains("$", regex = False)))
# combine polygon groups using conved hull
toponyms = revised\
    .loc[selection, ["png_filename", "groupid", "geometry"]]\
    .dissolve(by = ["png_filename", "groupid"], as_index = False)
toponyms["geometry"] = toponyms.geometry.convex_hull
toponyms["key"] =\
    toponyms.png_filename + toponyms.groupid.astype("string")

selection = revised\
    .loc[selection]\
    .sort_values(by = ["png_filename", "groupid", "wordid"], ascending = True)\
    .groupby(["png_filename", "groupid"], as_index = False)\
    .agg(text = ("word", " ".join), count = ("word", "count"))
selection["key"] =\
    selection.pop("png_filename") + selection.pop("groupid").astype("string")

toponyms  = pd.merge(
    toponyms, selection, how = "left", on = "key", indicator = True
)
assert (toponyms._merge != "both").sum() == 0, "Merge is not perfect"
toponyms.drop(columns = ["_merge", "key"], inplace = True)

In [ ]:
# Join toponyms and GB1900 gazetteer
toponym_gb1900 = gp.sjoin_nearest(
    toponyms,
    gb1900[["final_text", "geometry"]],
    how = "left",
    distance_col = "join_dist"
)
assert toponym_gb1900.index_right.isna().sum() == 0, "Not all records in left frame joined"
toponym_gb1900["text_match"] = (toponym_gb1900.text==toponym_gb1900.final_text)
toponym_gb1900.reset_index(drop = True, inplace = True)
toponym_gb1900.head()

In [ ]:
toponym_gb1900[["join_dist", "text_match"]]\
    .plot\
    .box(by = "text_match", vert = False)

In [ ]:
toponym_gb1900.loc[
    toponym_gb1900.join_dist > 50,
    ["png_filename", "groupid", "text", "final_text", "join_dist"]
]

In [ ]:
toponym_gb1900.loc[
    toponym_gb1900.text_match,
    ["png_filename", "groupid", "text", "final_text", "join_dist"]
]

Initial inspection shows that purely numerical values and OS Benchmark map features (these are features that start with (B.M.)) are not recorded Gb1900 gazetteer; these records should be removed.

Full stops are marked inconsistently in manual labels, because some of them do not appear on the binarized raster; recommend removing full-stops and commas from both manual label text and gb1900 text.

GB1900 includes unconvention characters such as "$\^{a}$"; these need to be normalised.